In [ ]:
# Set JAVA_HOME and HADOOP_HOME for Spark
import os
os.environ['JAVA_HOME'] = r'C:\tools\jdk-17.0.18+8'
os.environ['HADOOP_HOME'] = r'C:\tools\hadoop'
# Add both Java and Hadoop bin directories to PATH
os.environ['PATH'] = r'C:\tools\hadoop\bin;' + os.environ['JAVA_HOME'] + r'\bin;' + os.environ.get('PATH', '')

import pyspark
from pyspark.sql import SparkSession

print(f"JAVA_HOME: {os.environ['JAVA_HOME']}")
print(f"HADOOP_HOME: {os.environ['HADOOP_HOME']}")
hadoop_in_path = 'hadoop\\bin' in os.environ['PATH']
print(f"Hadoop bin in PATH: {hadoop_in_path}")

JAVA_HOME: C:\tools\jdk-17.0.18+8
HADOOP_HOME: C:\tools\hadoop


In [8]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [9]:
!curl -L -O https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

  6  123M    6 8252k    0     0  6996k      0  0:00:18  0:00:01  0:00:17 6996k
 16  123M   16 19.9M    0     0  9395k      0  0:00:13  0:00:02  0:00:11 11.9M
 25  123M   25 31.8M    0     0  10.0M      0  0:00:12  0:00:03  0:00:09 11.8M
 35  123M   35 43.5M    0     0  10.4M      0  0:00:11  0:00:04  0:00:07 11.8M
 44  123M   44 55.4M    0     0  10.7M      0  0:00:11  0:00:05  0:00:06 11.8M
 54  123M   54 67.2M    0     0  10.8M      0  0:00:11  0:00:06  0:00:05 11.8M
 63  123M   63 79.0M    0     0  11.0M      0  0:00:11  0:00:07  0:00:04 11.8M
 73  123M   73 90.9M    0     0  11.1M      0  0:0

In [10]:
# Read CSV with Spark (reads compressed .csv.gz directly)
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv.gz')

In [11]:
# Preview first few rows
df.show(5, truncate=False)

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|pickup_datetime    |dropoff_datetime   |PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|HV0003           |B02682              |2021-01-01 00:33:44|2021-01-01 00:49:07|230         |166         |NULL   |
|HV0003           |B02682              |2021-01-01 00:55:19|2021-01-01 01:18:21|152         |167         |NULL   |
|HV0003           |B02764              |2021-01-01 00:23:56|2021-01-01 00:38:05|233         |142         |NULL   |
|HV0003           |B02764              |2021-01-01 00:42:51|2021-01-01 00:45:50|142         |143         |NULL   |
|HV0003           |B02764              |2021-01-01 00:48:14|2021-01-01 01:08:42|143         |78          |NULL   |
+-----------------+--------------------+-------------------+-------------------+

In [12]:
# Count total rows
row_count = df.count()
print(f"Total rows: {row_count:,}")

Total rows: 11,908,468


In [13]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [14]:
# Extract first 1001 rows (1 header + 1000 data rows) to head.csv
import pandas as pd
df_head = pd.read_csv('fhvhv_tripdata_2021-01.csv.gz', nrows=1000)
df_head.to_csv('head.csv', index=False)
print(f"Saved {len(df_head)} rows to head.csv")

Saved 1000 rows to head.csv


In [15]:
import pandas as pd

In [16]:
df_pandas = pd.read_csv('head.csv')

In [17]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [18]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

Integer - 4 bytes
Long - 8 bytes

In [19]:
from pyspark.sql import types

In [20]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [21]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv.gz')

In [22]:
df = df.repartition(24)

In [ ]:
# Write parquet - using simple flat directory name
df.write.mode('overwrite').parquet('fhvhv_output')

In [ ]:
df = spark.read.parquet('fhvhv_output')

In [ ]:
df.printSchema()

SELECT * FROM df WHERE hvfhs_license_num =  HV0003

In [ ]:
from pyspark.sql import functions as F

In [ ]:
df.show()

In [ ]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [ ]:
crazy_stuff('B02884')

In [ ]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [ ]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

In [ ]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003')


In [ ]:
# Display first 10 rows
import pandas as pd
pd.read_csv('head.csv', nrows=10)